In [0]:
%pip install --quiet confluent-kafka

# **Test Spark Kafka connector**

In [0]:

# ============================================================
# BANGLAPAY | PHASE 1.1
# NATIVE SPARK KAFKA CONNECTOR SMOKE TEST
#
# Environment : DEV
# Source      : Confluent Kafka
# Topics      : 7
# Mode        : Bounded batch read
# Limit       : 5 messages per partition
#
# No message production.
# No offset commits.
# No source data modification.
# ============================================================

import json
import uuid

from confluent_kafka import Consumer, TopicPartition
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. Load DEV credentials
# ------------------------------------------------------------

SCOPE = "banglapay-ingestion"

bootstrap = dbutils.secrets.get(
    SCOPE, "CONFLUENT_BOOTSTRAP_SERVERS"
)

api_key = dbutils.secrets.get(
    SCOPE, "CONFLUENT_API_KEY"
)

api_secret = dbutils.secrets.get(
    SCOPE, "CONFLUENT_API_SECRET"
)

TOPICS = [
    "mfs.agent.events.v1",
    "mfs.customer.activity.v1",
    "mfs.merchant.events.v1",
    "mfs.risk.events.v1",
    "mfs.settlement.events.v1",
    "mfs.transaction.events.v1",
    "mfs.wallet.events.v1",
]

MAX_MESSAGES_PER_PARTITION = 5

# ------------------------------------------------------------
# 2. Get bounded offsets using Confluent metadata client
# ------------------------------------------------------------

probe = Consumer({
    "bootstrap.servers": bootstrap,
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": api_key,
    "sasl.password": api_secret,
    "group.id": f"banglapay-spark-probe-{uuid.uuid4()}",
    "enable.auto.commit": False
})

starting_offsets = {}
ending_offsets = {}

try:
    metadata = probe.list_topics(timeout=20)

    for topic in TOPICS:

        if topic not in metadata.topics:
            raise RuntimeError(f"Topic not found: {topic}")

        topic_metadata = metadata.topics[topic]

        if topic_metadata.error is not None:
            raise RuntimeError(
                f"Topic metadata error: {topic}"
            )

        starting_offsets[topic] = {}
        ending_offsets[topic] = {}

        for partition_id in sorted(topic_metadata.partitions):

            low, high = probe.get_watermark_offsets(
                TopicPartition(topic, partition_id),
                timeout=15,
                cached=False
            )

            start_offset = low
            end_offset = min(
                high,
                low + MAX_MESSAGES_PER_PARTITION
            )

            starting_offsets[topic][str(partition_id)] = (
                start_offset
            )

            ending_offsets[topic][str(partition_id)] = (
                end_offset
            )

finally:
    probe.close()

print("Kafka offset inventory completed.")
print("Topics:", len(TOPICS))
print("Partition count:", sum(
    len(value) for value in starting_offsets.values()
))

# ------------------------------------------------------------
# 3. Build Spark Kafka authentication configuration
# ------------------------------------------------------------

def jaas_escape(value):
    return (
        value.replace("\\", "\\\\")
             .replace('"', '\\"')
    )

jaas_config = (
    "kafkashaded.org.apache.kafka.common.security.plain."
    "PlainLoginModule required "
    f'username="{jaas_escape(api_key)}" '
    f'password="{jaas_escape(api_secret)}";'
)

spark_kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": jaas_config,
    "subscribe": ",".join(TOPICS),
    "startingOffsets": json.dumps(starting_offsets),
    "endingOffsets": json.dumps(ending_offsets)
}

# ------------------------------------------------------------
# 4. Read through native Spark Kafka connector
# ------------------------------------------------------------

raw_kafka_df = (
    spark.read
    .format("kafka")
    .options(**spark_kafka_options)
    .load()
)

# ------------------------------------------------------------
# 5. Display safe metadata only
# Never display original key/value bytes.
# ------------------------------------------------------------

preview_df = (
    raw_kafka_df
    .select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        F.length("key").alias("key_bytes"),
        F.length("value").alias("value_bytes"),
        F.hex(
            F.substring("value", 1, 5)
        ).alias("confluent_header_hex")
    )
    .orderBy("topic", "partition", "offset")
)

total_messages = raw_kafka_df.count()

print("\nSPARK KAFKA CONNECTOR: SUCCESS")
print("Messages read:", total_messages)

display(preview_df)

# ------------------------------------------------------------
# 6. Validate basic Confluent Avro wire headers
# Confluent framing: magic byte + 4-byte schema ID.
# ------------------------------------------------------------

header_check = (
    raw_kafka_df
    .select(
        "topic",
        F.hex(
            F.substring("value", 1, 1)
        ).alias("magic_byte")
    )
    .groupBy("topic", "magic_byte")
    .count()
    .orderBy("topic", "magic_byte")
)

display(header_check)

print("\nSPARK KAFKA SMOKE TEST: COMPLETED")

# Remove Python references to credential values.
del api_key, api_secret, jaas_config, spark_kafka_options